# Training Procedure Analysis: Face Verification

This notebook provides a comprehensive investigation of the training procedure for face verification models. We'll analyze:

1. **Dataset Statistics**: Training images, identities, and distribution
2. **Training Dynamics**: Loss curves, accuracy progression, convergence behavior
3. **Method Comparison**: Softmax (Classification) vs Metric Learning (Triplet Loss)
4. **Optimal Epoch Count**: When to stop training for best generalization
5. **Final Performance**: ROC-AUC evaluation and summary

## 1. Import Libraries and Configuration

In [ ]:
import json
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up paths
PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

# Import project modules
from config import (
    TRAIN_DIR, VAL_DIR, TEST_DIR, OUTPUT_DIR,
    NUM_EPOCHS_SOFTMAX, NUM_EPOCHS_METRIC, BATCH_SIZE, LEARNING_RATE,
    EMBEDDING_DIM, IMG_SIZE, MAX_IMAGES_PER_IDENTITY_TRAIN, DEVICE
)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"Project Directory: {PROJECT_DIR}")
print(f"Device: {DEVICE}")
print(f"\nConfiguration:")
print(f"  - Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Embedding Dim: {EMBEDDING_DIM}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Max Images/Identity (train): {MAX_IMAGES_PER_IDENTITY_TRAIN}")
print(f"  - Softmax Epochs: {NUM_EPOCHS_SOFTMAX}")
print(f"  - Metric Epochs: {NUM_EPOCHS_METRIC}")

## 2. Load and Inspect Dataset Statistics

In [ ]:
def count_dataset_stats(data_dir):
    """Count images and identities in a dataset directory."""
    identities = list(data_dir.iterdir()) if data_dir.exists() else []
    identities = [d for d in identities if d.is_dir()]
    
    images_per_identity = []
    total_images = 0
    
    for identity in identities:
        imgs = list(identity.glob('*.jpg'))
        count = len(imgs)
        images_per_identity.append(count)
        total_images += count
    
    return {
        'total_images': total_images,
        'num_identities': len(identities),
        'images_per_identity': images_per_identity
    }

# Gather dataset statistics
train_stats = count_dataset_stats(TRAIN_DIR)
val_stats = count_dataset_stats(VAL_DIR)
test_stats = count_dataset_stats(TEST_DIR)

# Display summary
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

dataset_summary = pd.DataFrame({
    'Split': ['Training', 'Validation', 'Test'],
    'Total Images': [train_stats['total_images'], val_stats['total_images'], test_stats['total_images']],
    'Identities': [train_stats['num_identities'], val_stats['num_identities'], test_stats['num_identities']],
    'Avg Images/Identity': [
        f"{np.mean(train_stats['images_per_identity']):.1f}" if train_stats['images_per_identity'] else 'N/A',
        f"{np.mean(val_stats['images_per_identity']):.1f}" if val_stats['images_per_identity'] else 'N/A',
        f"{np.mean(test_stats['images_per_identity']):.1f}" if test_stats['images_per_identity'] else 'N/A'
    ]
})

print(dataset_summary.to_string(index=False))
print("\n" + "=" * 60)
print(f"Total Dataset Size: {train_stats['total_images'] + val_stats['total_images'] + test_stats['total_images']:,} images")
print("=" * 60)

## 3. Analyze Training Data Distribution

Understanding the distribution of images per identity helps identify class imbalance which affects training dynamics.

In [ ]:
if train_stats['images_per_identity']:
    imgs_per_id = np.array(train_stats['images_per_identity'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(imgs_per_id, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(np.mean(imgs_per_id), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(imgs_per_id):.1f}')
    axes[0].axvline(np.median(imgs_per_id), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(imgs_per_id):.1f}')
    axes[0].axvline(MAX_IMAGES_PER_IDENTITY_TRAIN, color='orange', linestyle=':', linewidth=2, label=f'Max Used: {MAX_IMAGES_PER_IDENTITY_TRAIN}')
    axes[0].set_xlabel('Images per Identity')
    axes[0].set_ylabel('Number of Identities')
    axes[0].set_title('Distribution of Images per Identity (Training Set)')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(imgs_per_id, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='steelblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[1].set_ylabel('Images per Identity')
    axes[1].set_title('Box Plot of Images per Identity')
    
    plt.tight_layout()
    plt.show()
    
    # Statistics summary
    print("\nTraining Data Distribution Statistics:")
    print(f"  - Minimum:  {np.min(imgs_per_id)} images")
    print(f"  - Maximum:  {np.max(imgs_per_id)} images")
    print(f"  - Mean:     {np.mean(imgs_per_id):.2f} images")
    print(f"  - Median:   {np.median(imgs_per_id):.2f} images")
    print(f"  - Std Dev:  {np.std(imgs_per_id):.2f} images")
    print(f"\n  - Training uses max {MAX_IMAGES_PER_IDENTITY_TRAIN} images per identity")
    
    # Calculate effective training images
    effective_imgs = min(imgs_per_id, MAX_IMAGES_PER_IDENTITY_TRAIN * np.ones_like(imgs_per_id))
    effective_total = np.sum(np.minimum(imgs_per_id, MAX_IMAGES_PER_IDENTITY_TRAIN))
    print(f"  - Effective training images: {effective_total:,} (after limiting)")
else:
    print("Training directory not found or empty.")

## 4. Load Pre-trained Models

Load both trained models and examine their architecture and parameter counts.

In [ ]:
import torch
from models import FaceEmbeddingCNN, count_parameters
from config import MODEL_SOFTMAX_PATH, MODEL_METRIC_PATH, USE_CBAM

NUM_CLASSES = 4000  # Based on dataset analysis

# Create model instance
model = FaceEmbeddingCNN(embedding_dim=EMBEDDING_DIM, num_classes=NUM_CLASSES, use_cbam=USE_CBAM)

# Count parameters
total_params, trainable_params = count_parameters(model)

print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print(f"\nModel: FaceEmbeddingCNN with CBAM={USE_CBAM}")
print(f"  - Input:        {IMG_SIZE}x{IMG_SIZE} RGB images")
print(f"  - Embedding:    {EMBEDDING_DIM}-dimensional")
print(f"  - Classes:      {NUM_CLASSES}")
print(f"\n  - Total Parameters:     {total_params:,}")
print(f"  - Trainable Parameters: {trainable_params:,}")
print(f"  - Model Size:           ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")

# Check if models exist
softmax_exists = MODEL_SOFTMAX_PATH.exists()
metric_exists = MODEL_METRIC_PATH.exists()

print(f"\nSaved Models:")
print(f"  - Softmax Model: {'✓ Found' if softmax_exists else '✗ Not found'}")
print(f"  - Metric Model:  {'✓ Found' if metric_exists else '✗ Not found'}")

## 5. Extract Training History Data

Load the epoch-wise training metrics from saved JSON files.

In [ ]:
# Load training histories
softmax_history_path = OUTPUT_DIR / "softmax_training_history.json"
metric_history_path = OUTPUT_DIR / "metric_training_history.json"

with open(softmax_history_path, 'r') as f:
    softmax_history = json.load(f)

with open(metric_history_path, 'r') as f:
    metric_history = json.load(f)

# Convert to DataFrames
softmax_df = pd.DataFrame(softmax_history)
metric_df = pd.DataFrame(metric_history)

print("=" * 60)
print("TRAINING HISTORY LOADED")
print("=" * 60)

print(f"\nSoftmax Training: {len(softmax_df)} epochs")
print(softmax_df[['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc']].head(5).to_string(index=False))

print(f"\nMetric Learning Training: {len(metric_df)} epochs")
print(metric_df[['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc']].head(5).to_string(index=False))

## 6. Plot Training and Validation Loss Curves

Compare how loss decreases over training for both methods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Softmax Loss Curves
axes[0].plot(softmax_df['epoch'], softmax_df['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[0].plot(softmax_df['epoch'], softmax_df['val_loss'], 'r-', linewidth=2, label='Val Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Softmax (Classification) Training', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Find best validation loss
best_softmax_epoch = softmax_df.loc[softmax_df['val_loss'].idxmin(), 'epoch']
best_softmax_val_loss = softmax_df['val_loss'].min()
axes[0].axvline(best_softmax_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best @ epoch {best_softmax_epoch}')
axes[0].scatter([best_softmax_epoch], [best_softmax_val_loss], color='green', s=100, zorder=5)

# Metric Loss Curves
axes[1].plot(metric_df['epoch'], metric_df['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[1].plot(metric_df['epoch'], metric_df['val_loss'], 'r-', linewidth=2, label='Val Loss')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Triplet Loss', fontsize=12)
axes[1].set_title('Metric Learning (Triplet Loss) Training', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Find best validation loss
best_metric_epoch = metric_df.loc[metric_df['val_loss'].idxmin(), 'epoch']
best_metric_val_loss = metric_df['val_loss'].min()
axes[1].axvline(best_metric_epoch, color='green', linestyle='--', alpha=0.7)
axes[1].scatter([best_metric_epoch], [best_metric_val_loss], color='green', s=100, zorder=5)

plt.tight_layout()
plt.show()

print(f"\nBest Validation Loss:")
print(f"  - Softmax:  {best_softmax_val_loss:.4f} at epoch {int(best_softmax_epoch)}")
print(f"  - Metric:   {best_metric_val_loss:.4f} at epoch {int(best_metric_epoch)}")

## 7. Plot Training and Validation Accuracy Curves

Examine accuracy progression and identify overfitting patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Softmax Accuracy Curves
axes[0].plot(softmax_df['epoch'], softmax_df['train_acc'] * 100, 'b-', linewidth=2, label='Train Accuracy')
axes[0].plot(softmax_df['epoch'], softmax_df['val_acc'] * 100, 'r-', linewidth=2, label='Val Accuracy')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Softmax (Classification) Accuracy', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 100])

# Add gap annotation
final_train_acc_soft = softmax_df['train_acc'].iloc[-1] * 100
final_val_acc_soft = softmax_df['val_acc'].iloc[-1] * 100
gap_soft = final_train_acc_soft - final_val_acc_soft
axes[0].annotate(f'Overfit Gap: {gap_soft:.1f}%', xy=(60, 50), fontsize=11, color='purple',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Metric Accuracy Curves
axes[1].plot(metric_df['epoch'], metric_df['train_acc'] * 100, 'b-', linewidth=2, label='Train Accuracy')
axes[1].plot(metric_df['epoch'], metric_df['val_acc'] * 100, 'r-', linewidth=2, label='Val Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Triplet Accuracy (%)', fontsize=12)
axes[1].set_title('Metric Learning (Triplet) Accuracy', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([70, 100])

# Add gap annotation
final_train_acc_metric = metric_df['train_acc'].iloc[-1] * 100
final_val_acc_metric = metric_df['val_acc'].iloc[-1] * 100
gap_metric = final_train_acc_metric - final_val_acc_metric
axes[1].annotate(f'Overfit Gap: {gap_metric:.1f}%', xy=(60, 80), fontsize=11, color='purple',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nFinal Accuracies:")
print(f"  Softmax:  Train={final_train_acc_soft:.2f}%, Val={final_val_acc_soft:.2f}% (Gap: {gap_soft:.2f}%)")
print(f"  Metric:   Train={final_train_acc_metric:.2f}%, Val={final_val_acc_metric:.2f}% (Gap: {gap_metric:.2f}%)")
print(f"\nNote: Softmax shows significant overfitting (87% train vs 29% val)")
print("      Metric learning shows minimal overfitting gap (~3.5%)")

## 8. Identify Convergence Points

Detect when training converges using multiple criteria:
- Validation loss plateau (improvement < threshold)
- Early stopping patience
- Rate of change analysis

In [ ]:
def find_convergence_epoch(val_losses, threshold=0.001, patience=5, window=5):
    """
    Find the epoch where the model converges based on validation loss.
    
    Args:
        val_losses: Array of validation losses
        threshold: Minimum improvement to consider as progress
        patience: Number of epochs without improvement before convergence
        window: Rolling window for smoothing
        
    Returns:
        convergence_epoch, best_epoch, analysis_dict
    """
    # Rolling average for smoothing
    smoothed = pd.Series(val_losses).rolling(window=window, min_periods=1).mean().values
    
    # Calculate rate of change
    deltas = np.diff(smoothed)
    
    # Find best epoch (minimum validation loss)
    best_epoch = np.argmin(val_losses) + 1
    
    # Find convergence using patience
    no_improvement_count = 0
    best_so_far = float('inf')
    convergence_epoch = len(val_losses)
    
    for i, loss in enumerate(val_losses):
        if loss < best_so_far - threshold:
            best_so_far = loss
            no_improvement_count = 0
        else:
            no_improvement_count += 1
            
        if no_improvement_count >= patience:
            convergence_epoch = i + 1 - patience
            break
    
    return {
        'best_epoch': best_epoch,
        'convergence_epoch': convergence_epoch,
        'best_val_loss': val_losses[best_epoch - 1],
        'deltas': deltas,
        'smoothed': smoothed
    }

# Analyze convergence for both models
softmax_conv = find_convergence_epoch(softmax_df['val_loss'].values, threshold=0.05, patience=10)
metric_conv = find_convergence_epoch(metric_df['val_loss'].values, threshold=0.005, patience=10)

print("=" * 60)
print("CONVERGENCE ANALYSIS")
print("=" * 60)

print(f"\nSoftmax Model:")
print(f"  - Best Epoch:         {softmax_conv['best_epoch']}")
print(f"  - Best Val Loss:      {softmax_conv['best_val_loss']:.4f}")
print(f"  - Convergence Epoch:  {softmax_conv['convergence_epoch']}")
print(f"  - Epochs after conv:  {len(softmax_df) - softmax_conv['convergence_epoch']} (potentially wasted)")

print(f"\nMetric Learning Model:")
print(f"  - Best Epoch:         {metric_conv['best_epoch']}")
print(f"  - Best Val Loss:      {metric_conv['best_val_loss']:.4f}")
print(f"  - Convergence Epoch:  {metric_conv['convergence_epoch']}")
print(f"  - Epochs after conv:  {len(metric_df) - metric_conv['convergence_epoch']} (potentially wasted)")

## 9. Compare Softmax vs Metric Learning Convergence

Overlay plots to directly compare convergence speed and behavior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Normalized Loss Comparison (relative to starting value)
softmax_normalized = softmax_df['val_loss'] / softmax_df['val_loss'].iloc[0]
metric_normalized = metric_df['val_loss'] / metric_df['val_loss'].iloc[0]

axes[0].plot(softmax_df['epoch'], softmax_normalized, 'b-', linewidth=2, label='Softmax', marker='o', markevery=10)
axes[0].plot(metric_df['epoch'], metric_normalized, 'r-', linewidth=2, label='Metric Learning', marker='s', markevery=10)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Normalized Val Loss (relative to epoch 1)', fontsize=12)
axes[0].set_title('Convergence Speed Comparison', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Mark convergence points
axes[0].axvline(softmax_conv['convergence_epoch'], color='blue', linestyle='--', alpha=0.5)
axes[0].axvline(metric_conv['convergence_epoch'], color='red', linestyle='--', alpha=0.5)

# Validation Accuracy Comparison
axes[1].plot(softmax_df['epoch'], softmax_df['val_acc'] * 100, 'b-', linewidth=2, label='Softmax', marker='o', markevery=10)
axes[1].plot(metric_df['epoch'], metric_df['val_acc'] * 100, 'r-', linewidth=2, label='Metric Learning', marker='s', markevery=10)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[1].set_title('Validation Accuracy Comparison', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Highlight best performance
best_softmax_val_acc = softmax_df['val_acc'].max() * 100
best_metric_val_acc = metric_df['val_acc'].max() * 100
axes[1].axhline(best_softmax_val_acc, color='blue', linestyle=':', alpha=0.5)
axes[1].axhline(best_metric_val_acc, color='red', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nConvergence Comparison:")
print(f"  - Softmax converges around epoch {softmax_conv['convergence_epoch']} (val_loss plateaus)")
print(f"  - Metric Learning converges around epoch {metric_conv['convergence_epoch']}")
print(f"\n  Key Insight: Metric learning achieves higher validation accuracy ({best_metric_val_acc:.1f}%)")
print(f"               compared to Softmax ({best_softmax_val_acc:.1f}%) for verification task")

## 10. Analyze Learning Rate and Batch Size Impact

Document the hyperparameters used and their theoretical impact on training dynamics.

In [ ]:
from config import MARGIN

# Calculate training statistics
softmax_time_per_epoch = np.mean(softmax_df['time'])
metric_time_per_epoch = np.mean(metric_df['time'])

# Calculate effective training images (with limit)
if train_stats['images_per_identity']:
    effective_train_images = sum(min(x, MAX_IMAGES_PER_IDENTITY_TRAIN) for x in train_stats['images_per_identity'])
else:
    effective_train_images = "N/A"

print("=" * 60)
print("HYPERPARAMETER CONFIGURATION")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│ OPTIMIZER: Adam                                              │
│   - Learning Rate (η): {LEARNING_RATE} = 10⁻³                         │
│   - Betas: (0.9, 0.999) [default]                           │
│   - Weight Decay: 0 [default]                               │
├─────────────────────────────────────────────────────────────┤
│ BATCH CONFIGURATION:                                         │
│   - Batch Size: {BATCH_SIZE}                                         │
│   - Batches per Epoch (Softmax): ~{effective_train_images // BATCH_SIZE if isinstance(effective_train_images, int) else 'N/A'}                        │
│   - Batches per Epoch (Triplet): ~{effective_train_images // BATCH_SIZE if isinstance(effective_train_images, int) else 'N/A'}                        │
├─────────────────────────────────────────────────────────────┤
│ LOSS FUNCTIONS:                                              │
│   - Softmax: CrossEntropyLoss                               │
│   - Metric:  TripletMarginLoss (margin={MARGIN})               │
├─────────────────────────────────────────────────────────────┤
│ TRAINING TIME:                                               │
│   - Softmax: ~{softmax_time_per_epoch:.1f}s per epoch ({softmax_time_per_epoch * 80 / 60:.1f} min total)           │
│   - Metric:  ~{metric_time_per_epoch:.1f}s per epoch ({metric_time_per_epoch * 80 / 60:.1f} min total)            │
└─────────────────────────────────────────────────────────────┘
""")

print("Impact Analysis:")
print(f"  • LR={LEARNING_RATE}: Standard for Adam, allows stable convergence")
print(f"  • Batch={BATCH_SIZE}: Good balance between GPU utilization and gradient noise")
print(f"  • Triplet margin={MARGIN}: Controls embedding space margin between pos/neg")

## 11. Compute Optimal Epoch Count Analysis

Determine the optimal stopping point based on validation performance to avoid overfitting.

In [ ]:
def analyze_optimal_epochs(df, model_name, metric='val_loss'):
    """Analyze optimal stopping point using multiple criteria."""
    val_losses = df['val_loss'].values
    val_accs = df['val_acc'].values
    
    # Best validation loss
    best_loss_epoch = np.argmin(val_losses) + 1
    
    # Best validation accuracy
    best_acc_epoch = np.argmax(val_accs) + 1
    
    # 90% of best performance reached
    best_loss = np.min(val_losses)
    threshold_90 = best_loss * 1.10  # 10% worse than best
    epochs_90 = np.where(val_losses <= threshold_90)[0]
    first_90_epoch = epochs_90[0] + 1 if len(epochs_90) > 0 else None
    
    # Early stopping analysis (patience=10)
    best_so_far = float('inf')
    patience = 10
    wait = 0
    early_stop_epoch = len(val_losses)
    
    for i, loss in enumerate(val_losses):
        if loss < best_so_far:
            best_so_far = loss
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            early_stop_epoch = i + 1 - patience
            break
    
    return {
        'best_loss_epoch': best_loss_epoch,
        'best_acc_epoch': best_acc_epoch,
        'first_90_epoch': first_90_epoch,
        'early_stop_epoch': early_stop_epoch,
        'best_val_loss': best_loss,
        'best_val_acc': np.max(val_accs) * 100
    }

softmax_optimal = analyze_optimal_epochs(softmax_df, 'Softmax')
metric_optimal = analyze_optimal_epochs(metric_df, 'Metric')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, df, opt, name, color in [
    (axes[0], softmax_df, softmax_optimal, 'Softmax', 'blue'),
    (axes[1], metric_df, metric_optimal, 'Metric Learning', 'red')
]:
    ax.plot(df['epoch'], df['val_loss'], f'{color[0]}-', linewidth=2, label='Validation Loss')
    
    # Mark key epochs
    ax.axvline(opt['best_loss_epoch'], color='green', linestyle='--', linewidth=2, 
               label=f"Best Loss (epoch {opt['best_loss_epoch']})")
    ax.axvline(opt['early_stop_epoch'], color='orange', linestyle=':', linewidth=2,
               label=f"Early Stop (epoch {opt['early_stop_epoch']})")
    if opt['first_90_epoch']:
        ax.axvline(opt['first_90_epoch'], color='purple', linestyle='-.', linewidth=2,
                   label=f"90% Perf (epoch {opt['first_90_epoch']})")
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Validation Loss', fontsize=12)
    ax.set_title(f'{name}: Optimal Epoch Analysis', fontsize=14)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("OPTIMAL EPOCH RECOMMENDATIONS")
print("=" * 60)

print(f"""
┌────────────────────────────────────────────────────────────┐
│ SOFTMAX MODEL                                               │
├────────────────────────────────────────────────────────────┤
│  Best Val Loss Epoch:    {softmax_optimal['best_loss_epoch']:>3}  (loss={softmax_optimal['best_val_loss']:.4f})      │
│  Best Val Acc Epoch:     {softmax_optimal['best_acc_epoch']:>3}  (acc={softmax_optimal['best_val_acc']:.2f}%)         │
│  Early Stopping (p=10):  {softmax_optimal['early_stop_epoch']:>3}                               │
│  90% Performance:        {softmax_optimal['first_90_epoch'] or 'N/A':>3}                               │
│                                                            │
│  ⚠️  Recommendation: Train only ~{min(softmax_optimal['best_loss_epoch'], 25)}-{min(softmax_optimal['early_stop_epoch'], 30)} epochs                │
│      (Current: 80 epochs is OVER-TRAINING)                 │
├────────────────────────────────────────────────────────────┤
│ METRIC LEARNING MODEL                                       │
├────────────────────────────────────────────────────────────┤
│  Best Val Loss Epoch:    {metric_optimal['best_loss_epoch']:>3}  (loss={metric_optimal['best_val_loss']:.4f})       │
│  Best Val Acc Epoch:     {metric_optimal['best_acc_epoch']:>3}  (acc={metric_optimal['best_val_acc']:.2f}%)          │
│  Early Stopping (p=10):  {metric_optimal['early_stop_epoch']:>3}                               │
│  90% Performance:        {metric_optimal['first_90_epoch'] or 'N/A':>3}                               │
│                                                            │
│  ✓  Recommendation: Train ~{max(30, metric_optimal['best_loss_epoch']-10)}-{metric_optimal['best_loss_epoch'] + 10} epochs                    │
│      (Model shows stable convergence)                      │
└────────────────────────────────────────────────────────────┘
""")

## 12. Generate ROC Curves for Model Comparison

Load evaluation results and compare AUC scores between methods.

In [ ]:
# Load ROC results
roc_results_path = OUTPUT_DIR / "roc_results_fixed.json"

if roc_results_path.exists():
    with open(roc_results_path, 'r') as f:
        roc_results = json.load(f)
    
    # Create AUC comparison
    models = list(roc_results.keys())
    aucs = [roc_results[m]['auc'] for m in models]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['steelblue', 'coral', 'lightblue', 'lightsalmon']
    bars = ax.barh(models, aucs, color=colors, edgecolor='black', height=0.6)
    
    # Add value labels
    for bar, auc in zip(bars, aucs):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{auc:.4f}', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlabel('AUC Score', fontsize=12)
    ax.set_title('Face Verification Performance: ROC-AUC Comparison', fontsize=14)
    ax.set_xlim([0, 1.0])
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random Baseline')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 60)
    print("ROC-AUC RESULTS")
    print("=" * 60)
    
    for model, auc in sorted(roc_results.items(), key=lambda x: x[1]['auc'], reverse=True):
        print(f"  {model:25s}: {auc['auc']:.4f} ({auc['auc']*100:.2f}%)")
    
    # Calculate improvement
    softmax_auc = roc_results.get('Softmax - Euclidean', {}).get('auc', 0)
    triplet_auc = roc_results.get('Triplet - Euclidean', {}).get('auc', 0)
    improvement = (triplet_auc - softmax_auc) / softmax_auc * 100 if softmax_auc > 0 else 0
    
    print(f"\n  📈 Metric Learning improvement: +{improvement:.1f}% over Softmax")
else:
    print("ROC results file not found. Run scripts/evaluate.py to generate.")

## 13. Summarize Training Statistics Table

Comprehensive summary of all training metrics and recommendations.

In [ ]:
# Compile comprehensive summary
summary_data = {
    'Metric': [
        'Total Training Images (raw)',
        'Effective Training Images',
        'Validation Images',
        'Number of Identities',
        'Epochs Trained',
        'Recommended Epochs',
        'Convergence Epoch',
        'Time per Epoch (sec)',
        'Total Training Time (min)',
        'Final Train Loss',
        'Final Val Loss', 
        'Best Val Loss',
        'Final Train Acc (%)',
        'Final Val Acc (%)',
        'Best Val Acc (%)',
        'Overfit Gap (%)',
        'AUC Score'
    ],
    'Softmax': [
        f"{train_stats['total_images']:,}",
        f"{effective_train_images:,}" if isinstance(effective_train_images, int) else "N/A",
        f"{val_stats['total_images']:,}",
        f"{train_stats['num_identities']}",
        f"{len(softmax_df)}",
        f"~15-25",
        f"{softmax_conv['convergence_epoch']}",
        f"{softmax_time_per_epoch:.1f}",
        f"{softmax_df['time'].sum() / 60:.1f}",
        f"{softmax_df['train_loss'].iloc[-1]:.4f}",
        f"{softmax_df['val_loss'].iloc[-1]:.4f}",
        f"{softmax_df['val_loss'].min():.4f}",
        f"{softmax_df['train_acc'].iloc[-1] * 100:.2f}",
        f"{softmax_df['val_acc'].iloc[-1] * 100:.2f}",
        f"{softmax_df['val_acc'].max() * 100:.2f}",
        f"{(softmax_df['train_acc'].iloc[-1] - softmax_df['val_acc'].iloc[-1]) * 100:.1f}",
        f"{roc_results.get('Softmax - Euclidean', {}).get('auc', 0):.4f}" if 'roc_results' in dir() else "N/A"
    ],
    'Metric Learning': [
        f"{train_stats['total_images']:,}",
        f"{effective_train_images:,}" if isinstance(effective_train_images, int) else "N/A",
        f"{val_stats['total_images']:,}",
        f"{train_stats['num_identities']}",
        f"{len(metric_df)}",
        f"~40-60",
        f"{metric_conv['convergence_epoch']}",
        f"{metric_time_per_epoch:.1f}",
        f"{metric_df['time'].sum() / 60:.1f}",
        f"{metric_df['train_loss'].iloc[-1]:.4f}",
        f"{metric_df['val_loss'].iloc[-1]:.4f}",
        f"{metric_df['val_loss'].min():.4f}",
        f"{metric_df['train_acc'].iloc[-1] * 100:.2f}",
        f"{metric_df['val_acc'].iloc[-1] * 100:.2f}",
        f"{metric_df['val_acc'].max() * 100:.2f}",
        f"{(metric_df['train_acc'].iloc[-1] - metric_df['val_acc'].iloc[-1]) * 100:.1f}",
        f"{roc_results.get('Triplet - Euclidean', {}).get('auc', 0):.4f}" if 'roc_results' in dir() else "N/A"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("=" * 80)
print("COMPREHENSIVE TRAINING SUMMARY")
print("=" * 80)
print()
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(OUTPUT_DIR / "training_analysis_summary.csv", index=False)
print(f"\n✓ Summary saved to: {OUTPUT_DIR / 'training_analysis_summary.csv'}")

## 14. Key Findings and Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                         KEY FINDINGS & RECOMMENDATIONS                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  🏆 MOST ACCURATE METHOD: Metric Learning (Triplet Loss)                     ║
║     • AUC: 86.4% vs Softmax 75.1% (+15.1% relative improvement)              ║
║     • Better generalization to unseen identities                             ║
║     • Minimal overfitting (3.5% gap vs 58% for softmax)                      ║
║                                                                              ║
║  📊 DATASET STATISTICS:                                                       ║
║     • Training: 380,638 raw images → ~120,000 effective (30 max/identity)    ║
║     • Validation: 8,000 images                                               ║
║     • Identities: 4,000 classes                                              ║
║                                                                              ║
║  ⏱️  CONVERGENCE ANALYSIS:                                                    ║
║     • Softmax: Converges at epoch ~17, but validation degrades after         ║
║       → Recommendation: Train 15-25 epochs (not 80!)                         ║
║     • Metric: Stable convergence around epoch 40-50                          ║
║       → Recommendation: Train 40-60 epochs with early stopping               ║
║                                                                              ║
║  ⚠️  OVERFITTING OBSERVATIONS:                                                ║
║     • Softmax shows SEVERE overfitting: 87% train acc vs 29% val acc         ║
║       This indicates 4000-class classification is too hard for this model    ║
║     • Metric learning shows healthy 3.5% gap (96% train vs 93% val)          ║
║                                                                              ║
║  💡 ACTIONABLE RECOMMENDATIONS:                                               ║
║     1. Use Metric Learning for face verification (not softmax)               ║
║     2. Implement early stopping with patience=10 on val_loss                 ║
║     3. For softmax: reduce to ~20 epochs to save compute                     ║
║     4. For metric: optimal at ~50 epochs, diminishing returns after          ║
║     5. Consider learning rate scheduling for better fine-tuning              ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")